8.펑션콜링(function Calling)
- 에이전트 시스템을 개발하는 방법 중 하나
- LLM이 코드의 특정 함수를 사용자의 요구에 맞게 스스로 인식하고 호출하여 문제를 해결할 수 있도록 하는 기술
- LLM이 외부 시스템과 상호작용하거나 복잡한 작업을 처리할 수 있도록 설계된 프레임워크
- 함수 정의 > 도구 등록 > 함수 호출 실행 > 결과 처리 및 응답 생성
- ReAct에이전트 : 사고 - 행동 - 관찰
  . 장점 : 사고 과정을 통해 복잠한 문제 해결
  . 단점 : 루프 과정에서 반드시 사고 과정을 거쳐야 하므로, 다소 복잡해질 수 있고 답변에 도달하기까지 느려진다
  . 예시 : 챗봇, 실시간 물류 경로 최적화 등
- 펑션콜링 : 행동 - 관찰
  . 장점 : 구현이 간단하고 빠르다
  . 예시 : 항공권 예약, 날씨 정보 제공

In [9]:
from llama_index.llms.openai import OpenAI
from llama_index.core.tools import FunctionTool
from llama_index.core.agent import FunctionCallingAgentWorker, AgentRunner

llm = OpenAI(model="gpt-4o")

def add(a,b):
    """주어진 두 숫자를 더하고 결과를 출력합니다."""
    return a+b

def mul(a,b):
    """주어진 두 숫자를 곱하고 결과를 출력합니다."""
    return a*b

def div(a,b):
    """주어진 두 숫자를 나누고 결과를 출력합니다."""
    return a/b

at = FunctionTool.from_defaults(fn=add)
mt = FunctionTool.from_defaults(fn=mul)
dt = FunctionTool.from_defaults(fn=div)

agent_worker = FunctionCallingAgentWorker.from_tools(
    tools=[at, mt, dt],
    llm=llm,
    verbose=True, 
    allow_parallel_tool_calls=False
)

agent = AgentRunner(agent_worker)
response = agent.chat("(77*2)+2를 78로 나눈 값을 계산해줘")
print(response)

Added user message to memory: (77*2)+2를 78로 나눈 값을 계산해줘
=== Calling Function ===
Calling function: mul with args: {"a": 77, "b": 2}
=== Function Output ===
154
=== Calling Function ===
Calling function: add with args: {"a": 154, "b": 2}
=== Function Output ===
156
=== Calling Function ===
Calling function: div with args: {"a": 156, "b": 78}
=== Function Output ===
2.0
=== LLM Response ===
\((77 \times 2) + 2\)를 78로 나눈 값은 2.0입니다.
\((77 \times 2) + 2\)를 78로 나눈 값은 2.0입니다.


8.3 외부 API를 활용한 펑션 콜링
- 외부 API와 연동하여 실시간 데이터를 활용할 수 있도록 함으로써 RAG의 한계를 보완
- RAG에서 발생할 수 있는 검색 지연과 데이터 처리 시간을 효과적으로 줄일 수 있음
- RAG은 실시간으로 변동되는 정보에는 적합하지 않음

In [14]:
#증시 정보 호출 에이전트
import numpy as np
import yfinance as yf

!pip install yfinance==0.2.55
!pip install curl_cffi #파이썬에서 웹요청을 보낼떄 사용하는 라이브러리

from curl_cffi import requests

llm = OpenAI(model="gpt-4o")

def get_stock_price_us(code):
    session = requests.Session(impersonate="chrome")
    ticker = yf.Ticker(f"{code}", session=session)
    todays_data = ticker.history(period='1d')
    
    if not todays_data.empty:
        close_price = round(todays_data['Close'].iloc[0], 2)
        close_data = todays_data.index[0].strftime('%Y-%m-%d')
        return close_price, close_data
    return None, None

def get_stock_price_Korea(code):
    session = requests.Session(impersonate="chrome")
    ticker = yf.Ticker(f"{code}.KS", session=session)
    todays_data = ticker.history(period='1d')

    if not todays_data.empty:
        close_price = round(todays_data['Close'].iloc[0], 2)
        close_data = todays_data.index[0].strftime('%Y-%m-%d')
        return close_price, close_data
    return None, None

#도구 등록
stock_k = FunctionTool.from_defaults(fn=get_stock_price_Korea)
stock_u = FunctionTool.from_defaults(fn=get_stock_price_us)

#에이전트 생성
agent_worker = FunctionCallingAgentWorker.from_tools(
    [stock_k, stock_u],
    llm=llm,
    verbose=True, 
    allow_parallel_tool_calls=False
)

agent = AgentRunner(agent_worker)
response1 = agent.chat("TESLA 최신 종가가 어떻게 돼?")
print(response1)
response2 = agent.chat("삼성전자의 최신 종가가 어떻게 돼?")
print(response2)


Added user message to memory: TESLA 최신 종가가 어떻게 돼?
=== Calling Function ===
Calling function: get_stock_price_us with args: {"code": "TSLA"}
=== Function Output ===
(347.79, '2025-09-10')
=== LLM Response ===
TESLA의 최신 종가는 2025년 9월 10일 기준으로 $347.79입니다.
TESLA의 최신 종가는 2025년 9월 10일 기준으로 $347.79입니다.
Added user message to memory: 삼성전자의 최신 종가가 어떻게 돼?
=== Calling Function ===
Calling function: get_stock_price_Korea with args: {"code": "005930"}
=== Function Output ===
(73400.0, '2025-09-11')
=== LLM Response ===
삼성전자의 최신 종가는 2025년 9월 11일 기준으로 73,400원입니다.
삼성전자의 최신 종가는 2025년 9월 11일 기준으로 73,400원입니다.


8.4 펑션 콜링으로 구현하는 RAG 에이전트
- 복잡한 질의가 입력된 고난이도의 RAG 상황에서도 적절한 응답을 생성하는것이 가능

In [ ]:
#삼성전자 2022년부터 2024년까지 3개년 사업보고서 읽고 분석
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.settings import Settings
from llama_index.core import (SimpleDirectoryReader, VectorStoreIndex)
from llama_index.core.tools import QueryEngineTool

Settings.llm = OpenAI(model="gpt-4o")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

#데이터 다운로드
import os
import urllib.parse
import requests
import re

# GitHub의 blob URL을 그대로 받으면 HTML 페이지가 저장됩니다. 겉으론 ...pdf 파일처럼 보이지만 실제 내용은 HTML이라 PDF 리더에서 깨집니다.
# -> raw 붙이고 /blob/삭제 필요
urls = [
    "https://raw.github.com/llama-index-tutorial/llama-index-tutorial/main/ch08/data/%5B%EC%82%BC%EC%84%B1%EC%A0%84%EC%9E%90%5D%EC%82%AC%EC%97%85%EB%B3%B4%EA%B3%A0%EC%84%9C_2022.pdf",
    "https://raw.github.com/llama-index-tutorial/llama-index-tutorial/main/ch08/data/%5B%EC%82%BC%EC%84%B1%EC%A0%84%EC%9E%90%5D%EC%82%AC%EC%97%85%EB%B3%B4%EA%B3%A0%EC%84%9C_2023.pdf",
    "https://raw.github.com/llama-index-tutorial/llama-index-tutorial/main/ch08/data/%5B%EC%82%BC%EC%84%B1%EC%A0%84%EC%9E%90%5D%EC%82%AC%EC%97%85%EB%B3%B4%EA%B3%A0%EC%84%9C_2024.pdf"
]

#각 파일 다운로드
for url in urls:
    encoded_filenmae = url.split("/")[-1] #URL에서 파일명 추출
    decoded_filename = urllib.parse.unquote(encoded_filenmae) #한글 파일명 복원
    response = requests.get(url)

    if response.status_code == 200:
        #임시 파일명으로 저장
        temp_filename = "temp_download_file" + os.path.splitext(decoded_filename)[1]
        with open(temp_filename, 'wb') as f:
            f.write(response.content)
        os.rename(temp_filename, decoded_filename)
        print(f"완료:{decoded_filename} 다운로드 완료")
    else:
        print(f"오류:{url} 다운로드 실패(상태코드: {response.status_code})\n") 

완료:[삼성전자]사업보고서_2022.pdf 다운로드 완료
완료:[삼성전자]사업보고서_2023.pdf 다운로드 완료
완료:[삼성전자]사업보고서_2024.pdf 다운로드 완료


In [28]:
s2024_docs = SimpleDirectoryReader(input_files=["./data/[삼성전자]사업보고서_2024.pdf"]).load_data()
s2023_docs = SimpleDirectoryReader(input_files=["./data/[삼성전자]사업보고서_2023.pdf"]).load_data()
s2022_docs = SimpleDirectoryReader(input_files=["./data/[삼성전자]사업보고서_2022.pdf"]).load_data()

#인덱싱
s2024_index = VectorStoreIndex.from_documents(s2024_docs)
s2023_index = VectorStoreIndex.from_documents(s2023_docs)
s2022_index = VectorStoreIndex.from_documents(s2022_docs)

#쿼리엔진변환
s2024_engine = s2024_index.as_query_engine(similarity_top_k=3)
s2023_engine = s2023_index.as_query_engine(similarity_top_k=3)
s2022_engine = s2022_index.as_query_engine(similarity_top_k=3)

#도구등록
query_engine_tool = [
    QueryEngineTool.from_defaults(
        query_engine=s2024_index,
        name="samsung_2024",
        description=("삼성전자의 2024년 재무상태에 대해 정보 제공해 주세요. 도구에 입력할 떄 자세한 일반 텍스트 질문을 사용합니다. 실적 분석할 떄 보고서를 충분히 검토한 후 답변해 주세요.")
    ),
    QueryEngineTool.from_defaults(
        query_engine=s2023_index,
        name="samsung_2023",
        description=("삼성전자의 2023년 재무상태에 대해 정보 제공해 주세요. 도구에 입력할 떄 자세한 일반 텍스트 질문을 사용합니다. 실적 분석할 떄 보고서를 충분히 검토한 후 답변해 주세요.")
    ),
    QueryEngineTool.from_defaults(
        query_engine=s2022_index,
        name="samsung_2022",
        description=("삼성전자의 2022년 재무상태에 대해 정보 제공해 주세요. 도구에 입력할 떄 자세한 일반 텍스트 질문을 사용합니다. 실적 분석할 떄 보고서를 충분히 검토한 후 답변해 주세요.")
    ),
]

#펑션콜링 에이전트
from llama_index.core.agent import FunctionCallingAgentWorker, AgentRunner
agent_worker = FunctionCallingAgentWorker.from_tools(
    tools=query_engine_tool,
    llm=llm,
    verbose=True
)

agent = AgentRunner(agent_worker)
response1 = agent.chat("2022년 매출액과 2023년 매출액, 2024년 매출액을 차례로 알려줘")

Added user message to memory: 2022년 매출액과 2023년 매출액, 2024년 매출액을 차례로 알려줘
=== Calling Function ===
Calling function: samsung_2022 with args: {"input": "2022\ub144 \uc0bc\uc131\uc804\uc790\uc758 \ub9e4\ucd9c\uc561\uc5d0 \ub300\ud574 \uc54c\ub824\uc8fc\uc138\uc694."}
=== Function Output ===
Encountered error: 'VectorStoreIndex' object has no attribute 'query'
=== Calling Function ===
Calling function: samsung_2023 with args: {"input": "2023\ub144 \uc0bc\uc131\uc804\uc790\uc758 \ub9e4\ucd9c\uc561\uc5d0 \ub300\ud574 \uc54c\ub824\uc8fc\uc138\uc694."}
=== Function Output ===
Encountered error: 'VectorStoreIndex' object has no attribute 'query'
=== Calling Function ===
Calling function: samsung_2024 with args: {"input": "2024\ub144 \uc0bc\uc131\uc804\uc790\uc758 \ub9e4\ucd9c\uc561\uc5d0 \ub300\ud574 \uc54c\ub824\uc8fc\uc138\uc694."}
=== Function Output ===
Encountered error: 'VectorStoreIndex' object has no attribute 'query'
=== LLM Response ===
죄송합니다. 현재 삼성전자의 2022년, 2023년, 2024년 매출액 정보를 가져오는 데